# Visualization

In [0]:
%run ../config/config

In [0]:
import numpy as np
import pandas as pd
from pyspark.sql import functions as F
import plotly.graph_objects as go
from plotly.subplots import make_subplots

#### 1. Load Data

In [0]:
symbol = "VCB"  

cleaned_pdf = (
    spark.table(tables["cleaned_ohlcv"])
    .filter(F.col("symbol") == symbol)
    .orderBy("date")
    .toPandas()
)

labels_pdf = (
    spark.table(tables["phase_labels"])
    .filter(F.col("symbol") == symbol)
    .orderBy("date")
    .toPandas()
)

indicators_pdf = (
    spark.table(tables["technical_indicators"])
    .filter(F.col("symbol") == symbol)
    .orderBy("date")
    .toPandas()
)

predictions_pdf = (
    spark.table(tables["cnn_lstm_predictions"])
    .filter(F.col("symbol") == symbol)
    .orderBy("date")
    .select("date", "predicted", "actual")
    .toPandas()
)
predictions_pdf["date"] = pd.to_datetime(predictions_pdf["date"])

print(f"Symbol: {symbol}")
print(f"Price data: {len(cleaned_pdf)} rows")
print(f"Labels: {len(labels_pdf)} rows")
print(f"Predictions (test set): {len(predictions_pdf)} rows")

if cleaned_pdf.empty:
    print(f"No data found for symbol '{symbol}'.")

#### 2. Wyckoff sequence detection and candlestick visualization

In [0]:
min_run_days = 3             # 1 trading week
min_phase = 3                # at least 3 monotonic phases to count as a sequence
complete_window_bar = 120    # 6 month(trading days)


def phase_family(phase_id):
    return "accum" if phase_id < 5 else "distrib"


def phase_idx_in_family(phase_id):
    return phase_id % 5


def detect_sequences(phase_df, phase_col="predicted",
                     min_run_length=min_run_days,
                     min_sequence_phases=min_phase,
                     completion_window_bars=complete_window_bar):
    """
    Detect monotonic non-skipping Wyckoff sequences from a phase prediction time series.
    """
    df = phase_df.sort_values("date").reset_index(drop=True).copy()
    if df.empty:
        return []

    # Step 1: run length code
    runs = []
    cur_phase = int(df.iloc[0][phase_col])
    cur_start = 0
    for i in range(1, len(df)):
        if int(df.iloc[i][phase_col]) != cur_phase:
            runs.append({"phase": cur_phase, "start": cur_start, "end": i - 1})
            cur_phase = int(df.iloc[i][phase_col])
            cur_start = i
    runs.append({"phase": cur_phase, "start": cur_start, "end": len(df) - 1})

    # Step 2: drop runs shorter than min_run_days, absorb into previous run
    smoothed = []
    for run in runs:
        length = run["end"] - run["start"] + 1
        if length >= min_run_length:
            smoothed.append(dict(run))
        elif smoothed:
            smoothed[-1]["end"] = run["end"]

    # Step 3: detect monotonic non-skipping sequences
    sequences = []
    cur_seq = None
    for run in smoothed:
        family = phase_family(run["phase"])
        idx = phase_idx_in_family(run["phase"])

        if cur_seq is None:
            cur_seq = {"family": family, "runs": [run]}
            continue

        last_phase = cur_seq["runs"][-1]["phase"]
        same_family = (family == cur_seq["family"])
        last_idx = phase_idx_in_family(last_phase)

        if same_family and idx == last_idx + 1:
            cur_seq["runs"].append(run)
        elif same_family and idx == last_idx:
            cur_seq["runs"][-1]["end"] = run["end"]  
        else:
            if len(cur_seq["runs"]) >= min_sequence_phases:
                sequences.append(cur_seq)
            cur_seq = {"family": family, "runs": [run]}
    if cur_seq and len(cur_seq["runs"]) >= min_sequence_phases:
        sequences.append(cur_seq)

    # Step 4: annotate
    for seq in sequences:
        seq["phases"] = [r["phase"] for r in seq["runs"]]
        seq["phase_names"] = [phase_names[p] for p in seq["phases"]]
        seq["start_bar_idx"] = seq["runs"][0]["start"]
        seq["end_bar_idx"] = seq["runs"][-1]["end"]
        seq["start_date"] = pd.to_datetime(df.iloc[seq["start_bar_idx"]]["date"])
        seq["end_date"] = pd.to_datetime(df.iloc[seq["end_bar_idx"]]["date"])
        seq["n_bars"] = seq["end_bar_idx"] - seq["start_bar_idx"] + 1
        reached_phases = [phase_idx_in_family(p) for p in seq["phases"]]
        seq["complete"] = (4 in reached_phases) and (seq["n_bars"] <= completion_window_bars)
        seq["highest_phase_idx"] = max(reached_phases)

    return sequences

In [0]:
# Apply sequence detection to test-set predictions
test_sequences = detect_sequences(predictions_pdf, phase_col="predicted")

print(f"Detected {len(test_sequences)} sequence(s) in test period (2024) for {symbol}:")
for i, seq in enumerate(test_sequences):
    short_names = " - ".join(p.split("_")[-1] for p in seq["phase_names"])
    status = "complete" if seq["complete"] else f"partial ({len(seq['phases'])}/5)"
    print(f"  #{i+1} [{seq['family']:>7s}] {seq['start_date'].date()} → {seq['end_date'].date()} "
          f"({seq['n_bars']:>3} bars): {short_names}  [{status}]")

In [0]:
def plot_candlestick_with_sequences(price_df, sequences, symbol_str,
                                    start_date=None, end_date=None,
                                    completion_table=None,
                                    predictions_df=None, labels_df=None):
    
    df = price_df.copy()
    df["date"] = pd.to_datetime(df["date"])
    if start_date:
        df = df[df["date"] >= pd.to_datetime(start_date)]
    if end_date:
        df = df[df["date"] <= pd.to_datetime(end_date)]
    if df.empty:
        print("No price data in the selected range.")
        return None

    # Merge per-bar phase info for hover tooltip
    df = df.sort_values("date").reset_index(drop=True)
    if predictions_df is not None and not predictions_df.empty:
        pred_df = predictions_df[["date", "predicted"]].copy()
        pred_df["date"] = pd.to_datetime(pred_df["date"])
        df = df.merge(pred_df, on="date", how="left")
        df["predicted_name"] = df["predicted"].map(phase_names).fillna("-")
    else:
        df["predicted_name"] = "-"
    if labels_df is not None and not labels_df.empty:
        lab_df = labels_df[["date", "phase_label"]].copy()
        lab_df["date"] = pd.to_datetime(lab_df["date"])
        df = df.merge(lab_df, on="date", how="left")
        df["actual_name"] = df["phase_label"].map(phase_names).fillna("-")
    else:
        df["actual_name"] = "-"

    fig = make_subplots(
        rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.03,
        row_heights=[0.75, 0.25],
        subplot_titles=[f"{symbol_str} - Candlestick + Predicted Wyckoff Sequences", "Volume"],
    )

    fig.add_trace(go.Candlestick(
        x=df["date"], open=df["open"], high=df["high"], low=df["low"], close=df["close"],
        name="OHLC", increasing_line_color="green", decreasing_line_color="red",
        customdata=np.stack([df["predicted_name"], df["actual_name"]], axis=-1),
        hovertext=[
            f"<b>{d.strftime('%Y-%m-%d')}</b><br>"
            f"Open: {o:,.0f}  High: {h:,.0f}<br>"
            f"Low: {l:,.0f}  Close: {c:,.0f}<br>"
            f"<b>Predicted phase:</b> {pn}<br>"
            f"<b>Actual phase:</b> {an}"
            for d, o, h, l, c, pn, an in zip(
                df["date"], df["open"], df["high"], df["low"], df["close"],
                df["predicted_name"], df["actual_name"],
            )
        ],
        hoverinfo="text",
    ), row=1, col=1)

    vol_colors = ["green" if c >= o else "red" for c, o in zip(df["close"], df["open"])]
    fig.add_trace(go.Bar(
        x=df["date"], y=df["volume"], marker_color=vol_colors, opacity=0.5,
        name="Volume", showlegend=False,
    ), row=2, col=1)

    # Sequence boxes - dashed outline, no fill, label above
    price_max = df["high"].max()
    price_min = df["low"].min()
    label_y = price_max * 1.02

    for i, seq in enumerate(sequences):
        in_range_seq = (seq["end_date"] >= df["date"].min()) and (seq["start_date"] <= df["date"].max())
        if not in_range_seq:
            continue

        family = seq["family"]
        outline = "rgba(0,150,0,0.85)" if family == "accum" else "rgba(200,30,30,0.85)"
        family_label = "ACCUM" if family == "accum" else "DISTRIB"
        short_names = "-".join(p.split("_")[-1] for p in seq["phase_names"])
        status = "complete" if seq["complete"] else f"partial({len(seq['phases'])}/5)"

        fig.add_shape(
            type="rect",
            x0=seq["start_date"], x1=seq["end_date"],
            y0=price_min * 0.98, y1=price_max * 1.01,
            line=dict(color=outline, width=2, dash="dash"),
            fillcolor="rgba(0,0,0,0)",
            layer="above",
            row=1, col=1,
        )

        annot_text = f"{family_label} #{i+1}: {short_names} [{status}]"
        if completion_table is not None and not seq["complete"]:
            highest = seq["highest_phase_idx"]
            base_rate = completion_table.get(family, {}).get(highest, {})
            if base_rate.get("n", 0) > 0:
                annot_text += f"<br>P(complete | reached {chr(ord('A')+highest)}) = {base_rate['p']:.2f} [n={base_rate['n']}]"

        fig.add_annotation(
            x=seq["start_date"], y=label_y, text=annot_text,
            showarrow=False, xanchor="left", yanchor="bottom",
            font=dict(size=9, color=outline), align="left",
            row=1, col=1,
        )

    fig.update_layout(
        xaxis_rangeslider_visible=False, width=1300, height=720,
        hovermode="x unified",
        legend=dict(orientation="h", yanchor="bottom", y=1.06, xanchor="right", x=1),
    )
    fig.update_xaxes(rangebreaks=[dict(bounds=["sat", "mon"])])
    return fig

#### 3. Sequence completion probability

In [0]:
def _wilson_ci(k, n, z=1.96):
    if n == 0:
        return (0.0, 0.0)
    p_hat = k / n
    denom = 1 + z * z / n
    center = (p_hat + z * z / (2 * n)) / denom
    margin = z * np.sqrt((p_hat * (1 - p_hat) + z * z / (4 * n)) / n) / denom
    return (max(0.0, center - margin), min(1.0, center + margin))


def compute_completion_base_rates(train_label_df, label_col="phase_label",
                                  completion_window_bars=complete_window_bar):
    
    # Reuse detect_sequences logic (rename column to "predicted")
    df_in = train_label_df.rename(columns={label_col: "predicted"})
    seqs = detect_sequences(df_in, phase_col="predicted",
                            completion_window_bars=completion_window_bars)

    table = {"accum": {}, "distrib": {}}
    for family in ["accum", "distrib"]:
        family_seqs = [s for s in seqs if s["family"] == family]
        for x in range(5):  # 0=A ... 4=E
            reached_x = [s for s in family_seqs if s["highest_phase_idx"] >= x]
            completed = [s for s in reached_x if s["complete"]]
            n = len(reached_x)
            k = len(completed)
            ci_lo, ci_hi = _wilson_ci(k, n)
            table[family][x] = {"n": n, "k": k,
                                "p": (k / n) if n > 0 else 0.0,
                                "ci_lo": ci_lo, "ci_hi": ci_hi}
    return table


# Load training-period actual labels
train_labels_pdf = (
    spark.table(tables["phase_labels"])
    .filter(F.col("date") <= train_end)
    .select("symbol", "date", "phase_label")
    .toPandas()
)
train_labels_pdf["date"] = pd.to_datetime(train_labels_pdf["date"])

# Per-symbol sequence detection then aggregate 
all_train_seqs_table = {"accum": {x: {"n": 0, "k": 0} for x in range(5)},
                        "distrib": {x: {"n": 0, "k": 0} for x in range(5)}}
for sym, sym_df in train_labels_pdf.groupby("symbol", sort=False):
    sym_table = compute_completion_base_rates(sym_df.reset_index(drop=True))
    for fam in ["accum", "distrib"]:
        for x in range(5):
            all_train_seqs_table[fam][x]["n"] += sym_table[fam][x]["n"]
            all_train_seqs_table[fam][x]["k"] += sym_table[fam][x]["k"]

# Recompute p + CI from aggregated counts
completion_table = {"accum": {}, "distrib": {}}
for fam in ["accum", "distrib"]:
    for x in range(5):
        n = all_train_seqs_table[fam][x]["n"]
        k = all_train_seqs_table[fam][x]["k"]
        ci_lo, ci_hi = _wilson_ci(k, n)
        completion_table[fam][x] = {"n": n, "k": k,
                                    "p": (k / n) if n > 0 else 0.0,
                                    "ci_lo": ci_lo, "ci_hi": ci_hi}

# Display as table
print(f"Complete base rates: Training period ({train_start} to {train_end}), all symbols")
print(f"Window: complete = reach E within {complete_window_bar} bars of sequence start")
print(f"{'Family':<10} {'Conditioning':<20} {'P(complete)':<14} {'95% Wilson CI':<22} {'n':<6}")
for fam in ["accum", "distrib"]:
    for x in range(4):  # exclude x=4 (E)
        row = completion_table[fam][x]
        cond = f"highest reached = {chr(ord('A')+x)}"
        ci_str = f"[{row['ci_lo']:.3f}, {row['ci_hi']:.3f}]"
        print(f"{fam:<10} {cond:<20} {row['p']:.3f}{'':<8} {ci_str:<22} {row['n']:<6}")
    print()

#### 4. Candlestick + Sequences with completion probability annotations

In [0]:
fig = plot_candlestick_with_sequences(
    cleaned_pdf, test_sequences, symbol,
    start_date=test_start, end_date=test_end,
    completion_table=completion_table,
    predictions_df=predictions_pdf,
    labels_df=labels_pdf,
)
if fig:
    displayHTML(fig.to_html())

#### 5. EDA Summary Statistics

In [0]:
print(f"Summary - {symbol}")

print(f"\nPrice Data:")
print(f"  Date range: {cleaned_pdf['date'].min()} to {cleaned_pdf['date'].max()}")
print(f"  Trading days: {len(cleaned_pdf):,}")
print(f"  Close range: {cleaned_pdf['close'].min():.0f} → {cleaned_pdf['close'].max():.0f}")
print(f"  Avg daily volume: {cleaned_pdf['volume'].mean():,.0f}")

if not labels_pdf.empty:
    print(f"\nPhase Distribution for {symbol}:")
    phase_dist = labels_pdf["phase_name"].value_counts()
    for phase, count in phase_dist.items():
        pct = count / len(labels_pdf) * 100
        print(f"  {phase}: {count:,} ({pct:.1f}%)")

    print(f"\nAvg confidence: {labels_pdf['phase_confidence'].mean():.3f}")
    print(f"Trading ranges detected: {labels_pdf['tr_id'].nunique()}")